In [1]:
from datasets import load_dataset, load_from_disk
from transformers import LEDConfig, DataCollatorForSeq2Seq, pipeline, PreTrainedTokenizerBase, LEDTokenizerFast, LEDForConditionalGeneration
import evaluate
import numpy as np
from peft import LoraConfig, get_peft_model
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from peft import get_peft_model, LoraConfig, TaskType, PeftModel, prepare_model_for_kbit_training
from tqdm import tqdm
import json
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4"
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
print(torch.cuda.device_count())
from torch.utils.data import DataLoader
from safetensors.torch import load_file
import math

from typing import List, Dict, Tuple, Any, Optional
import warnings
warnings.filterwarnings('ignore')

import logging as pylogging
from transformers import logging

logging.set_verbosity_warning()

pylogging.getLogger("transformers.models.led.modeling_led").setLevel(pylogging.ERROR)

/home/23ucs642/miniconda3/envs/myenv/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/23ucs642/miniconda3/envs/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1


### Hierarchical Evaluation

In [2]:
MODEL_NAME = "allenai/led-base-16384"
CHECKPOINT_DIR = "/home/23ucs642/Video_Summarization_Translation/led_hier_adapter_run"
HIER_ADAPTER_PATH = os.path.join(CHECKPOINT_DIR, "hier_adapter.pth")
MAX_INPUT = 8192
MAX_TARGET = 516
DEFAULT_BATCH_SIZE = 6
NUM_PROC = 8
PAD_TO_MULTIPLE_OF = 1024
FP16 = True
DEVICE = 'cuda'

In [3]:
class AttentionPooling(nn.Module):
    def __init__(self, d_hidden: int, temperature: float = 1.0):
        super().__init__()
        self.scorer = nn.Linear(d_hidden, 1)
        nn.init.normal_(self.scorer.weight, std=0.02)
        if self.scorer.bias is not None:
            nn.init.zeros_(self.scorer.bias)
        self.temperature = float(temperature)

    def forward(self, tokens: torch.Tensor):
        # tokens: (T, d)
        if tokens.numel() == 0:
            return tokens.new_zeros(tokens.size(-1))
        scores = self.scorer(tokens).squeeze(-1)  # (T,)
        scores = scores / max(self.temperature, 1e-6)
        weights = F.softmax(scores, dim=0)       # (T,)
        pooled = (weights.unsqueeze(-1) * tokens).sum(dim=0)
        return pooled

In [4]:
class HierarchicalAdapter(nn.Module):

    def __init__(self, d_model:int, d_bottleneck:int=256, pooling:str="attn", tau:float=0.1, use_proj=True):
        super().__init__()
        assert pooling in ("attn","mean")
        self.d_model = d_model
        self.d_bottleneck = d_bottleneck
        self.tau = tau
        self.pooling = pooling

        self.W_down = nn.Linear(d_model, d_bottleneck)
        self.layernorm = nn.LayerNorm(d_bottleneck)
        self.activation = F.relu

        if pooling == "attn":
            self.attn_pool = AttentionPooling(d_bottleneck, temperature=1.0)
        else:
            self.attn_pool = None

        if use_proj:
            self.W_sim_q = nn.Linear(d_bottleneck, d_bottleneck, bias=True)
            self.W_sim_k = nn.Linear(d_bottleneck, d_bottleneck, bias=True)
        else:
            self.W_sim_q = None
            self.W_sim_k = None

        self.W_sim_v = nn.Linear(d_bottleneck, d_bottleneck, bias=True)
        self.W_up = nn.Linear(d_bottleneck, d_model)
        if self.W_up.bias is not None:
            nn.init.zeros_(self.W_up.bias)

    def _pool_span(self, span_tokens: torch.Tensor):
        if span_tokens.size(0) == 0:
            return span_tokens.new_zeros(self.d_bottleneck)
        if self.pooling == "attn":
            return self.attn_pool(span_tokens)
        return span_tokens.mean(dim=0)

    def forward(self, hidden_states: torch.Tensor, utterance_boundaries: Optional[List[List[Tuple[int,int]]]] = None):

        B, L, _ = hidden_states.shape
        device = hidden_states.device
        if utterance_boundaries is None:
            utterance_boundaries = [[(0, L)] for _ in range(B)]

        # 1. down project
        h_down = self.W_down(hidden_states)  # (B, L, d_b)
        h_down = self.layernorm(h_down)
        h_down = self.activation(h_down)

        # 2. build utt vectors
        utt_vectors = []
        mask_list = []
        for b in range(B):
            spans = utterance_boundaries[b] if len(utterance_boundaries[b])>0 else [(0, L)]
            per_utt = []
            for (s,e) in spans:
                s_c = max(0, min(s, L))
                e_c = max(s_c, min(e, L))
                if e_c <= s_c:
                    per_utt.append(torch.zeros(self.d_bottleneck, device=device))
                else:
                    span_tokens = h_down[b, s_c:e_c]  # (T, d_b)
                    pooled = self._pool_span(span_tokens)
                    per_utt.append(pooled)
            if len(per_utt) == 0:
                uv = torch.zeros(1, self.d_bottleneck, device=device)
                mask = torch.tensor([0], device=device)
            else:
                uv = torch.stack(per_utt, dim=0)  # (U_b, d_b)
                mask = torch.ones(uv.size(0), device=device)
            utt_vectors.append(uv)
            mask_list.append(mask)

        # 3. pad to max_U
        max_U = max(uv.size(0) for uv in utt_vectors)
        padded_uv = []
        masks = []
        for uv, mask in zip(utt_vectors, mask_list):
            U = uv.size(0)
            if U < max_U:
                pad = torch.zeros(max_U-U, self.d_bottleneck, device=device)
                padded = torch.cat([uv, pad], dim=0)
                m = torch.cat([mask, torch.zeros(max_U-U, device=device)], dim=0)
            else:
                padded = uv
                m = mask
            padded_uv.append(padded)
            masks.append(m)
        utterance_tensor = torch.stack(padded_uv, dim=0)  # (B, max_U, d_b)
        mask_tensor = torch.stack(masks, dim=0).to(utterance_tensor.device)  # (B, max_U)

        # 4. similarity / attention
        Q = self.W_sim_q(utterance_tensor) if self.W_sim_q is not None else utterance_tensor
        K = self.W_sim_k(utterance_tensor) if self.W_sim_k is not None else utterance_tensor
        V = self.W_sim_v(utterance_tensor)

        scores = torch.matmul(Q, K.transpose(-1,-2)) / math.sqrt(self.d_bottleneck)  # (B, max_U, max_U)
        # avoid fp16 overflow: use safe min for dtype
        min_val = torch.finfo(scores.dtype).min
        scores = scores.masked_fill(mask_tensor.unsqueeze(1)==0, min_val)
        attn = F.softmax(scores / max(self.tau, 1e-6), dim=-1)
        context = torch.matmul(attn, V)  # (B, max_U, d_b)

        # 5. scatter back to tokens (additive)
        h_up_tokens = h_down.clone()
        for b in range(B):
            spans = utterance_boundaries[b]
            U_b = min(len(spans), context.size(1))
            for u in range(U_b):
                s,e = spans[u]
                s_c = max(0, min(s, L))
                e_c = max(s_c, min(e, L))
                if e_c <= s_c:
                    continue
                ctx = context[b, u]  # (d_b,)
                h_up_tokens[b, s_c:e_c] = h_up_tokens[b, s_c:e_c] + ctx.unsqueeze(0)

        # 6. up-project & resid
        h_out = self.W_up(h_up_tokens)  # (B, L, d_model)
        return hidden_states + h_out

def attach_hierarchical_adapters_to_led(model: LEDForConditionalGeneration, d_bottleneck=256, pooling='attn', tau=0.1):

    if hasattr(model, "led") and hasattr(model.led, "encoder"):
        encoder = model.led.encoder
    elif hasattr(model, "model") and hasattr(model.model, "led") and hasattr(model.model.led, "encoder"):
        encoder = model.model.led.encoder
    elif hasattr(model, "model") and hasattr(model.model, "encoder"):
        encoder = model.model.encoder
    else:
        raise RuntimeError("Cannot locate LED encoder")

    for i, layer in enumerate(encoder.layers):
        adapter = HierarchicalAdapter(d_model=model.config.d_model, d_bottleneck=d_bottleneck, pooling=pooling, tau=tau)
        layer.add_module("hierarchical_adapter", adapter)



def apply_lora_peft_to_led(model: LEDForConditionalGeneration, r=8, alpha=32, dropout=0.0, target_modules=None):
    if target_modules is None:
        target_modules = ["q_proj", "k_proj", "v_proj", "out_proj", "k_proj_global", "v_proj_global"]

    lora_config = LoraConfig(
        r=r,
        lora_alpha=alpha,
        target_modules=target_modules,
        lora_dropout=dropout,
        bias="none",
        task_type="SEQ_2_SEQ_LM"
    )
    peft_model = get_peft_model(model, lora_config)
    return peft_model

In [5]:
def save_peft_and_hier(model: torch.nn.Module, checkpoint_dir: str):
    """
    Save PEFT (LoRA) adapters using model.save_pretrained(checkpoint_dir) if available,
    then also save hierarchical adapter params into checkpoint_dir/hier_adapter.pth
    so they can be reloaded later together with PeftModel.from_pretrained.
    - model: should be PeftModel (or the wrapped model); hierarchical adapters must be attached.
    """
    os.makedirs(checkpoint_dir, exist_ok=True)

    try:
        model.save_pretrained(checkpoint_dir)
    except Exception:
        # Try to fallback to calling base_model save if attribute exists
        if hasattr(model, "base_model") and hasattr(model.base_model, "save_pretrained"):
            model.base_model.save_pretrained(os.path.join(checkpoint_dir, "base_model"))
        else:
            # best-effort: save full model (heavy)
            torch.save(model.state_dict(), os.path.join(checkpoint_dir, "full_model_state.pth"))

    hier_state = {}
    for k, v in model.state_dict().items():
        if "hierarchical_adapter" in k:
            hier_state[k] = v.cpu()
    hier_path = os.path.join(checkpoint_dir, "hier_adapter.pth")
    torch.save(hier_state, hier_path)
    print(f"[save_peft_and_hier] Saved hierarchical adapter state ({len(hier_state)} tensors) -> {hier_path}")


def load_peft_and_hier(base_model: LEDForConditionalGeneration, checkpoint_dir: str):
    
    model = PeftModel.from_pretrained(base_model, checkpoint_dir, is_trainable=False)
    print("[load_peft_and_hier] Loaded PeftModel (LoRA) from", checkpoint_dir)

    hier_path = os.path.join(checkpoint_dir, "hier_adapter.pth")
    if os.path.exists(hier_path):
        hier_state = torch.load(hier_path, map_location="cpu")
        # We do a non-strict load so missing keys are allowed.
        missing = []
        unexpected = []
        model_state = model.state_dict()
        # filter hierarchical keys to model
        to_load = {k: v for k, v in hier_state.items() if k in model_state}
        if len(to_load) == 0:
            print("[load_peft_and_hier] No matching hierarchical adapter keys found in model.state_dict(). Did you attach adapters before loading?")
        else:
            model.load_state_dict(to_load, strict=False)
            print(f"[load_peft_and_hier] Loaded {len(to_load)} hierarchical adapter tensors from {hier_path}")

    else:
        print("[load_peft_and_hier] No hierarchical adapter file found at", hier_path)

    return model



def list_hier_keys(model: torch.nn.Module, limit: int = 30):
    keys = [k for k in model.state_dict().keys() if "hierarchical_adapter" in k]
    print("Found hierarchical adapter keys:", len(keys))
    for k in keys[:limit]:
        print("  ", k)
    if len(keys) > limit:
        print("  ... (and more)")

In [6]:
def build_utterance_boundaries_from_segments(segments, tokenizer, max_length=MAX_INPUT):
    tokenized = [tokenizer(s, add_special_tokens=False)["input_ids"] for s in segments]
    input_ids = []
    boundaries = []
    start = 0
    for ids in tokenized:
        if len(ids) == 0:
            continue
        end = start + len(ids)
        if end > max_length:
            remaining = max_length - start
            if remaining <= 0:
                break
            ids = ids[:remaining]
            end = start + len(ids)
            input_ids.extend(ids)
            boundaries.append((start, end))
            break
        input_ids.extend(ids)
        boundaries.append((start, end))
        start = end
    return input_ids, boundaries


In [7]:
def tokenize_example_with_segments_for_map(example, tokenizer, max_input=MAX_INPUT, max_target=MAX_TARGET):
    segments = []
    if "transcript_segments" in example and isinstance(example["transcript_segments"], dict):
        segments = example["transcript_segments"].get("text", [])
    if not segments and "transcript" in example:
        segments = [example["transcript"]]

    input_ids, boundaries = build_utterance_boundaries_from_segments(segments, tokenizer, max_length=max_input)
    if len(input_ids) == 0 and "transcript" in example:
        enc = tokenizer(example["transcript"], add_special_tokens=False, truncation=True, max_length=max_input)
        input_ids = enc["input_ids"]
        boundaries = [(0, len(input_ids))] if len(input_ids) > 0 else []

    target_text = example.get("summary") or example.get("abstract") or example.get("summary_text") or ""
    label_enc = tokenizer(target_text, truncation=True, max_length=max_target, padding=False)
    labels = label_enc["input_ids"]
    return {"input_ids": input_ids, "labels": labels, "utterance_boundaries": boundaries}


In [8]:
import globals_store
class CollatorWithUtteranceBoundaries:
    def __init__(self, tokenizer, pad_to_multiple_of=None):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, batch):
        globals_store.GLOBAL_UTTERANCE_BOUNDARIES = [ex.get("utterance_boundaries", [(0, len(ex.get("input_ids", [])))]) for ex in batch]

        input_ids_list = [ex.get("input_ids", []) for ex in batch]
        labels_list = [ex.get("labels", []) for ex in batch]

        max_len = max((len(x) for x in input_ids_list), default=0)
        if self.pad_to_multiple_of is not None and max_len % self.pad_to_multiple_of != 0:
            max_len = ((max_len // self.pad_to_multiple_of) + 1) * self.pad_to_multiple_of
        padded_inputs = [ids + [self.tokenizer.pad_token_id] * (max_len - len(ids)) for ids in input_ids_list]
        attention_mask = [[1] * len(ids) + [0] * (max_len - len(ids)) for ids in input_ids_list]

        max_label_len = max((len(l) for l in labels_list), default=0)
        padded_labels = [l + [-100] * (max_label_len - len(l)) for l in labels_list]

        batch_output = {
            "input_ids": torch.tensor(padded_inputs, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(padded_labels, dtype=torch.long),
        }
        return batch_output

In [9]:
dataset = load_from_disk("/home/23ucs642/Video_Summarization_Translation/tib_dataset")

train_dataset = dataset["train"]
valid_dataset = dataset["valid"]
test_dataset = dataset["test"]

In [10]:
out_dir = "/home/23ucs642/Video_Summarization_Translation/led_hier_adapter_run"
checkpoint_dir = "/home/23ucs642/Video_Summarization_Translation/led_hier_adapter_run/checkpoint-500"
model_name = "allenai/led-base-16384"
tokenized_disk = os.path.join(out_dir, "tokenized")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
per_device_eval_batch_size = 1
num_workers = 4
num_beams = 4
max_target_len = 516


checkpoint_dir = "/home/23ucs642/Video_Summarization_Translation/led_hier_adapter_run"
model_name = "allenai/led-base-16384"
base_model = LEDForConditionalGeneration.from_pretrained("allenai/led-base-16384")
attach_hierarchical_adapters_to_led(base_model, d_bottleneck=256, pooling="attn", tau=0.1)

model = load_peft_and_hier(base_model, checkpoint_dir)
tokenizer = LEDTokenizerFast.from_pretrained("allenai/led-base-16384")
model.eval()


[load_peft_and_hier] Loaded PeftModel (LoRA) from /home/23ucs642/Video_Summarization_Translation/led_hier_adapter_run
[load_peft_and_hier] Loaded 84 hierarchical adapter tensors from /home/23ucs642/Video_Summarization_Translation/led_hier_adapter_run/hier_adapter.pth


PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): LEDForConditionalGeneration(
      (led): LEDModel(
        (shared): Embedding(50265, 768, padding_idx=1)
        (encoder): LEDEncoder(
          (embed_tokens): Embedding(50265, 768, padding_idx=1)
          (embed_positions): LEDLearnedPositionalEmbedding(16384, 768)
          (layers): ModuleList(
            (0-5): 6 x LEDEncoderLayer(
              (self_attn): LEDEncoderAttention(
                (longformer_self_attn): LEDEncoderSelfAttention(
                  (query): Linear(in_features=768, out_features=768, bias=True)
                  (key): Linear(in_features=768, out_features=768, bias=True)
                  (value): Linear(in_features=768, out_features=768, bias=True)
                  (query_global): Linear(in_features=768, out_features=768, bias=True)
                  (key_global): Linear(in_features=768, out_features=768, bias=True)
                  (value_global): Linear(in_features=768, out_features

In [11]:
sample_split = list(dataset.keys())[0]
first_example = dataset[sample_split][0]
tokenized = dataset.map(
        lambda ex: tokenize_example_with_segments_for_map(ex, tokenizer, max_input=MAX_INPUT, max_target=MAX_TARGET),
        remove_columns=dataset[sample_split].column_names,
        num_proc=NUM_PROC,
    )

In [12]:
tokenized["train"].set_format(type="python")
eval_ds = tokenized["valid"]
train_ds = tokenized["train"]
test_ds = tokenized["test"]

data_collator = CollatorWithUtteranceBoundaries(tokenizer, pad_to_multiple_of=PAD_TO_MULTIPLE_OF)
loader = DataLoader(test_ds, batch_size=6, collate_fn = data_collator, num_workers=4)
device = DEVICE
model.to(device)

PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): LEDForConditionalGeneration(
      (led): LEDModel(
        (shared): Embedding(50265, 768, padding_idx=1)
        (encoder): LEDEncoder(
          (embed_tokens): Embedding(50265, 768, padding_idx=1)
          (embed_positions): LEDLearnedPositionalEmbedding(16384, 768)
          (layers): ModuleList(
            (0-5): 6 x LEDEncoderLayer(
              (self_attn): LEDEncoderAttention(
                (longformer_self_attn): LEDEncoderSelfAttention(
                  (query): Linear(in_features=768, out_features=768, bias=True)
                  (key): Linear(in_features=768, out_features=768, bias=True)
                  (value): Linear(in_features=768, out_features=768, bias=True)
                  (query_global): Linear(in_features=768, out_features=768, bias=True)
                  (key_global): Linear(in_features=768, out_features=768, bias=True)
                  (value_global): Linear(in_features=768, out_features

In [ ]:
rouge = evaluate.load("rouge")
preds = []
refs = []

print("Starting generation on test set (batches: {})".format(len(loader)))

with torch.no_grad():
    for batch in tqdm(loader, desc="Evaluating in batches"):
        # Collator produced torch tensors for input_ids/attention_mask/labels
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # The collator set GLOBAL_UTT_BOUNDS globally for the monkeypatched adapters.
        # Generate summaries
        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=max_target_len,
            num_beams=num_beams,
            early_stopping=True,
            no_repeat_ngram_size=3,
            repetition_penalty=2.5,     
            encoder_no_repeat_ngram_size=3,
            min_length=50,  
            
        )

        # decode predictions
        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        preds.extend(decoded)

        # decode references: labels have -100 for padding; convert to list of ids first
        labels = batch["labels"].cpu().numpy()
        for lab in labels:
            lab_ids = [int(x) for x in lab if x != -100]
            refs.append(tokenizer.decode(lab_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True))


Starting generation on test set (batches: 152)


Evaluating in batches:   1%|▍                                                           | 1/152 [00:10<26:41, 10.61s/it]

In [15]:
rouge = evaluate.load("rouge")
print("\nComputing ROUGE scores...")
results = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
results = {k: round(v * 100, 2) for k, v in results.items()}

print("\n==== ROUGE Evaluation Results ====")
for k, v in results.items():
    print(f"{k}: {v}")


Computing ROUGE scores...

==== ROUGE Evaluation Results ====
rouge1: 27.3
rouge2: 4.88
rougeL: 15.18
rougeLsum: 15.18


In [ ]:
refs[0], preds[0]

### LoRA Evaluation

In [2]:
MAX_INPUT = 8192
MAX_TARGET = 516
model_name = "allenai/led-base-16384"
model_dir = "/home/23ucs642/Video_Summarization_Translation/led_tib_lora/checkpoint-9000"
base_model = LEDForConditionalGeneration.from_pretrained(model_name)
tokenizer = LEDTokenizerFast.from_pretrained(model_name)
model = PeftModel.from_pretrained(base_model, model_dir)
dataset = load_from_disk("/home/23ucs642/Video_Summarization_Translation/tib_dataset")
test_dataset = dataset["test"]
model.eval()

PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): LEDForConditionalGeneration(
      (led): LEDModel(
        (shared): Embedding(50265, 768, padding_idx=1)
        (encoder): LEDEncoder(
          (embed_tokens): Embedding(50265, 768, padding_idx=1)
          (embed_positions): LEDLearnedPositionalEmbedding(16384, 768)
          (layers): ModuleList(
            (0-5): 6 x LEDEncoderLayer(
              (self_attn): LEDEncoderAttention(
                (longformer_self_attn): LEDEncoderSelfAttention(
                  (query): Linear(in_features=768, out_features=768, bias=True)
                  (key): Linear(in_features=768, out_features=768, bias=True)
                  (value): Linear(in_features=768, out_features=768, bias=True)
                  (query_global): Linear(in_features=768, out_features=768, bias=True)
                  (key_global): Linear(in_features=768, out_features=768, bias=True)
                  (value_global): Linear(in_features=768, out_features

In [ ]:
from torch.utils.data import DataLoader
print(test_dataset)
def collate_fn(batch):
    inputs = [ex["transcript"] for ex in batch]
    refs = [ex["abstract"] for ex in batch]
    model_inputs = tokenizer(
        inputs,
        return_tensors="pt",
        padding="longest",
        truncation=True,
        max_length=MAX_INPUT,
    )
    return model_inputs, refs

loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_fn,
)

all_predictions = []
all_references = []

model.eval()
model.to("cuda")

for inputs, labels in tqdm(loader, desc="Evaluating in batches"):
    inputs = {k: v.to("cuda") for k, v in inputs.items()}
    with torch.no_grad():
        generated_ids = base_model.generate(
            **inputs,
            max_length=MAX_TARGET,
            num_beams=2,                
            length_penalty=1.0,
            early_stopping=True,
            no_repeat_ngram_size=3,    
            repetition_penalty=2.5,     
            encoder_no_repeat_ngram_size=3,
            min_length=50,               
        )

    preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    all_predictions.extend(preds)
    all_references.extend(labels)

Dataset({
    features: ['doi', 'title', 'url', 'video_url', 'license', 'subject', 'genre', 'release_year', 'author', 'contributors', 'abstract', 'transcript', 'transcript_segments', 'keyframes', 'language'],
    num_rows: 911
})


Evaluating in batches:   1%|▌                                                           | 1/114 [00:18<34:50, 18.50s/it]

In [ ]:
all_predictions[0], all_references[0]

In [ ]:
rouge = evaluate.load("rouge")
print("\nComputing ROUGE scores...")
results = rouge.compute(predictions=all_predictions, references=all_references, use_stemmer=True)
results = {k: round(v * 100, 2) for k, v in results.items()}

print("\n==== ROUGE Evaluation Results ====")
for k, v in results.items():
    print(f"{k}: {v}")
